# Echo — Local-First Opportunity Engine | Gemma 4 Hackathon

Echo starts from a simple belief: most people are talented, but only some discover it because someone happened to notice, name, and package their work at the right time.

This notebook shows a live Echo run: Gemma 4 turns scattered effort into structured proof, opportunity readiness, daily practice, offline continuity, and an eval-gated Unsloth training path.

Run it two ways:

- **Kaggle-local full runtime:** leave `ECHO_BASE_URL` empty and attach the Gemma 4 model input; the notebook bootstraps vLLM + Echo.
- **Hosted Echo runtime:** set `ECHO_BASE_URL` to a running Echo backend for the fastest judge walkthrough.

---

_Built for the Gemma 4 Hackathon with Google's Gemma 4 E2B, Unsloth LoRA training, LiteRT-LM on-device inference, and a complete opportunity loop._


## Part 1 — Vision & Problem

### The Problem: Under-Observed Talent

Most people are not talentless. They are **under-observed**.

A first-generation student in a low-connectivity town builds a working prototype, helps classmates debug circuits, and keeps a detailed cost comparison spreadsheet. None of that is visible to a scholarship committee unless it becomes inspectable proof. Someone with time, context, and taste would say: *"That is evidence. Package it."*

Most people never get that level of attention. Echo tries to make that attention available as a local-first product loop.

### The Echo Loop

Echo is not a chatbot. It is a **closed-loop opportunity engine**:

```text
Signal -> Pattern Map -> Next Proof Step -> Outcome -> Proof Card -> Direction
```

Each stage feeds the next:

| Stage | What happens | Key Gemma 4 role |
| --- | --- | --- |
| **Talk** | User shares work, blockers, or questions | Long-context reasoning with memory |
| **Pattern Map** | Echo synthesises a working thesis from evidence | Multi-turn reasoning + structured output |
| **Practice** | Daily rep chosen from missing proof gaps | Tool calling -> `log_outcome` |
| **Proof** | Proof Camera turns artifacts into structured evidence | Multimodal extraction -> `create_proof_item` |
| **Outcome** | Practice results are logged and scored | Structured output -> engagement signal |
| **Home Brain Adapter** | User feedback + outcomes -> eval-gated Unsloth LoRA training | Shadow Clone pipeline |
| **Opportunity** | Readiness score + missing proof gap -> goal path | Opportunity engine |

### What This Notebook Covers

| Part | Section |
| --- | --- |
| 2 | Submission links |
| 3 | System architecture — 3 runtime modes, request flow, component stack |
| 4 | Why Gemma 4 specifically |
| 5 | Bootstrap & connect to live Echo backend |
| 6 | Noor's Story — full product demo (Proof Camera -> Opportunity) |
| 7 | Shadow Clone training deep dive — 4 clone types, JSONL formats, Unsloth config |
| 8 | Offline continuity & privacy boundary |
| 9 | Full feature contract & final report |
| 10 | Developer reference — setup, env vars, quick start |
| 11 | Hackathon track alignment |
| 12 | Demo script & video guide |


## Part 2 — Submission Links

## Submission Links

These are the public assets judges should use. The code repo and source notebook are already stable; add the final Kaggle and video URLs after publishing the final version.

| Asset | Link | Status |
| --- | --- | --- |
| Public code repo | `https://github.com/klei30/echo` | Public |
| Notebook source | `https://github.com/klei30/echo/blob/main/kaggle/echo_gemma4_good_demo.ipynb` | Public source of this demo |
| Kaggle notebook | Add final Kaggle URL after Save Version | Required before submission |
| 3-minute video | Add public YouTube / Drive URL | Required before submission |
| Live demo / Home Brain tunnel | Optional public runtime URL | Bonus path; notebook runtime remains the reliable judged path |
| Mobile/desktop screenshots | Add media gallery URL or repo folder | Recommended |

Judges should be able to understand the project from the notebook alone, inspect the code in GitHub, and feel the product through the video. Live mode is useful, but the full runtime path is the reliable demo path.


## Part 3 — System Architecture

### 3 Runtime Modes

Echo routes every request through one of three model layers depending on what is available:

```
┌─────────────────────────────────────────────────────────────────┐
│                     Echo Mobile / Web App                       │
│          Talk  │  Today  │  Passport  │  Proof Camera           │
└──────────────────────────┬──────────────────────────────────────┘
                           │
            ┌──────────────┼──────────────────┐
            │              │                  │
  ┌─────────▼─────┐  ┌────▼──────────┐  ┌───▼───────────────┐
  │  Home Brain   │  │  Cloud Echo   │  │   This Device     │
  │               │  │               │  │                   │
  │ Gemma 4 E2B   │  │ Remote model  │  │ LiteRT-LM Gemma   │
  │ via vLLM      │  │ Full memory   │  │ 4 E2B on Android  │
  │ Personal LoRA │  │ No training   │  │ Synced mem pack   │
  │ Full memory   │  │               │  │ No internet need  │
  │ + Training    │  │               │  │                   │
  └───────────────┘  └───────────────┘  └───────────────────┘
       GPU required       Always on         Fully offline
       Most private      Most available     Most portable
```

### Request Flow

1. Mobile app sends request to **Echo API** (`:8002`)
2. Echo checks `/v1/runtime/capabilities` → picks best available runtime
3. **Home Brain path**: Echo API → vLLM (`:8003`) with Gemma 4 E2B + optional personal LoRA adapter
4. **Cloud Echo path**: Echo API → remote Gemma 4 endpoint with user's cloud memory
5. **This Device path**: LiteRT-LM runs directly on Android with a synced memory pack (`.litertlm` format)
6. Response flows back through Echo's memory/proof engine before returning to the app
7. Engagement signals (deep, thumbs_up, thumbs_down) are collected → fed into the Shadow Clone pipeline

### Component Stack

| Component | Technology | Role |
| --- | --- | --- |
| Mobile app | Flutter | Talk, Today, Passport, Proof Camera UI |
| Echo API | FastAPI (Python) | Auth, memory, proof, opportunities, training state |
| Model runtime (Home Brain) | vLLM + Gemma 4 E2B | Serving with LoRA hot-swap |
| Model runtime (This Device) | LiteRT-LM | On-device `.litertlm` Gemma 4 E2B |
| Memory store | SQLite / Mem0 | Private memories, rules, engagement history |
| Training pipeline | Unsloth + TRL | SFT / DPO LoRA adapter training |
| Proof engine | Echo API `/v1/proof/*` | Artifact extraction, evidence, privacy filter |
| Opportunity engine | Echo API `/v1/opportunities` | Readiness scoring, missing-proof gaps |
| Offline pack | Echo API `/v1/offline/export` | Compressed memory + rules for This Device |

## Part 4 — Why Gemma 4

## Why Gemma 4 Matters Here

This is the technical claim the notebook must prove. Echo is not just wrapping an LLM around a chat box;
it uses Gemma 4 as the reasoning layer inside a full opportunity loop.

| Gemma 4 capability | Echo use in this demo | Evidence cell |
| --- | --- | --- |
| Multimodal understanding | Proof Camera reads artifacts, notes, screenshots, feedback, and test logs. | Proof Camera chapter |
| Structured output | Gemma returns proof JSON with evidence, skills, privacy risk, confidence, and missing context. | `vision_result` |
| Native tool/function calling | Gemma chooses `create_proof_item`, `log_outcome`, `generate_opportunity`, or `ask_for_feedback`. | `tool_call` |
| Long-context reasoning | Echo combines memories, outcomes, proof, daily check-ins, and current read. | Current Read + Today |
| Edge/offline deployment | This Device uses a compact `.litertlm` Gemma memory pack when away from Home Brain. | Offline export + this-device continuity check |
| Personal LoRA adaptation | Home Brain improves through eval-gated Unsloth LoRA training from user feedback. | Shadow Clone deep dive (Part 7) |
| Privacy-first compute | Gemma 4 E2B fits on a consumer GPU; no cloud required for Home Brain mode. | Architecture (Part 3) |

**Why E2B specifically?**  
The 2B parameter variant fits comfortably in 8 GB VRAM (bf16), meaning a single RTX 3060 or equivalent
can run vLLM + Echo simultaneously. This is the hardware most students and independent developers actually own.
The same checkpoint also compiles to LiteRT-LM format for on-device Android inference — one model family
spanning cloud, Home Brain, and offline phone.

> **Important honesty boundary:** this notebook runs a bounded real Unsloth LoRA demo loop by default
> when Kaggle has a GPU, dependencies, and a local Gemma 4 model path. The full production clone
> tournament remains optional via `ECHO_TRAINING_PROFILE=full`.

## Part 5 — Bootstrap & Connect

## Full Runtime Bootstrap

Default mode runs everything inside Kaggle: clone the public Echo repo, start vLLM, start Echo, then seed a public-safe demo user. Set `GEMMA4_MODEL_PATH` or `KAGGLE_GEMMA4_MODEL_PATH` to the attached Gemma 4 model directory. Only set `ECHO_BASE_URL` when you intentionally want hosted Echo mode instead of Kaggle-local execution.

In [ ]:
import json
import os
os.environ.setdefault("MPLBACKEND", "Agg")
import socket
import subprocess
import sys
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
try:
    from IPython.display import Markdown, display
except Exception:
    class Markdown(str):
        pass
    def display(value):
        print(value)

EXTERNAL_ECHO_BASE_URL = os.getenv("ECHO_BASE_URL", "").rstrip("/")
BASE_URL = EXTERNAL_ECHO_BASE_URL
TOKEN = os.getenv("ECHO_TOKEN", "")
GITHUB_REPO = os.getenv("ECHO_GITHUB_REPO") or os.getenv("GITHUB_REPO", "https://github.com/klei30/echo")
DEMO_SEED_TOKEN = os.getenv("ECHO_DEMO_SEED_TOKEN", "kaggle-demo-seed")
USE_DEMO_SEED = os.getenv("ECHO_USE_DEMO_SEED", "1").lower() not in {"0", "false", "no"}

# Full-runtime Kaggle controls. By default, no ECHO_BASE_URL means bootstrap all services inside Kaggle.
BOOTSTRAP_DEFAULT = "0" if EXTERNAL_ECHO_BASE_URL else "1"
BOOTSTRAP_FULL = os.getenv("ECHO_BOOTSTRAP_FULL", BOOTSTRAP_DEFAULT).lower() not in {"0", "false", "no"}
INSTALL_DEPS_DEFAULT = "1" if (not EXTERNAL_ECHO_BASE_URL and Path("/kaggle").exists()) else "0"
INSTALL_DEPS = os.getenv("ECHO_INSTALL_DEPS", INSTALL_DEPS_DEFAULT).lower() in {"1", "true", "yes"}

def default_echo_workdir():
    if Path("/kaggle").exists():
        return Path("/kaggle/working/echo")
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "main.py").exists() and (candidate / "kaggle").exists():
            return candidate
    return Path.cwd()

ECHO_WORKDIR = Path(os.getenv("ECHO_WORKDIR") or default_echo_workdir()).resolve()
VLLM_BASE_URL = os.getenv("GEMMA4_VLLM_BASE_URL", "http://127.0.0.1:8003/v1").rstrip("/")
GEMMA4_MODEL_ID = os.getenv("GEMMA4_BASE_MODEL", "gemma4_e2b")
GEMMA4_MAX_MODEL_LEN = os.getenv("GEMMA4_MAX_MODEL_LEN", "8192")

def default_local_gemma_model_path():
    candidates = [
        Path.home() / ".cache" / "huggingface" / "hub" / "models--unsloth--gemma-4-E2B-it" / "snapshots",
        Path.home() / ".cache" / "huggingface" / "hub" / "models--google--gemma-4-E2B-it" / "snapshots",
    ]
    for snapshots in candidates:
        if snapshots.exists():
            dirs = sorted([p for p in snapshots.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
            if dirs:
                return str(dirs[0])
    return ""

GEMMA4_MODEL_PATH = (
    os.getenv("GEMMA4_MODEL_PATH")
    or os.getenv("KAGGLE_GEMMA4_MODEL_PATH")
    or os.getenv("KAGGLE_MODEL_PATH")
    or default_local_gemma_model_path()
)

def detect_running_vllm_model_root():
    try:
        response = requests.get(f"{VLLM_BASE_URL}/models", timeout=5)
        response.raise_for_status()
        models = response.json().get("data", [])
        for model in models:
            if model.get("id") == GEMMA4_MODEL_ID and model.get("root"):
                return model.get("root")
        return models[0].get("root") if models else ""
    except Exception:
        return ""

RUNNING_VLLM_MODEL_ROOT = detect_running_vllm_model_root()
EFFECTIVE_GEMMA_MODEL_SOURCE = GEMMA4_MODEL_PATH or RUNNING_VLLM_MODEL_ROOT

LIVE = False
USER_ID = ""
CALL_LOG = []
LIVE_ERRORS = []
PROCESSES = []
RUN_DASHBOARD = []


def now_iso():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def log_step(message):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)


def md(text):
    display(Markdown(text))


def show_json(title, data, limit=6000):
    md(f"### {title}")
    text = json.dumps(data, indent=2, ensure_ascii=False)
    if len(text) > limit:
        text = text[:limit] + "\n... truncated ..."
    display(Markdown(f"```json\n{text}\n```"))


def show_table(title, rows):
    md(f"### {title}")
    df = pd.DataFrame(rows)
    display(df)
    return df


def record_step(step, status, evidence="", detail=""):
    RUN_DASHBOARD.append({"step": step, "status": status, "evidence": evidence, "detail": detail})


def show_run_dashboard(title="Run dashboard"):
    return show_table(title, RUN_DASHBOARD)


def tail_log(path, lines=30):
    p = Path(path)
    if not p.exists():
        return f"missing log: {p}"
    try:
        return "\n".join(p.read_text(encoding="utf-8", errors="replace").splitlines()[-lines:])
    except Exception as exc:
        return f"could not read {p}: {exc}"


def show_process_logs(title="Process log tails", lines=30):
    logs = {p["name"]: tail_log(p["log"], lines=lines) for p in PROCESSES if p.get("log")}
    show_json(title, logs, limit=12000)
    return logs


def compact_training_rows(training_result):
    data = training_result.get("data", training_result) if isinstance(training_result, dict) else {}
    runtime_steps = data.get("runtime_steps", {}) if isinstance(data, dict) else {}
    promotion = data.get("promotion", {}) if isinstance(data, dict) else {}
    dataset = data.get("dataset", {}) if isinstance(data, dict) else {}
    eval_data = promotion.get("eval", {}) if isinstance(promotion, dict) else {}
    return [
        {"check": "status", "value": data.get("status"), "why_it_matters": "training reached a terminal state"},
        {"check": "real_training", "value": data.get("real_training"), "why_it_matters": "proves Unsloth actually ran"},
        {"check": "rows_written", "value": dataset.get("rows_written"), "why_it_matters": "shows live user signals became JSONL"},
        {"check": "max_steps", "value": (data.get("bounds") or {}).get("max_steps"), "why_it_matters": "bounded for Kaggle runtime"},
        {"check": "vLLM stopped", "value": runtime_steps.get("stopped_vllm_for_training"), "why_it_matters": "frees GPU memory for LoRA"},
        {"check": "vLLM restarted", "value": runtime_steps.get("restarted_vllm"), "why_it_matters": "Home Brain recovered after training"},
        {"check": "hot_swap_ok", "value": promotion.get("hot_swap_ok"), "why_it_matters": "adapter was loaded into serving runtime"},
        {"check": "eval_passed", "value": eval_data.get("passed"), "why_it_matters": "adapter is eval-gated before promotion"},
    ]


def top_readiness(opportunities_response):
    items = opportunities_response.get("items", []) if isinstance(opportunities_response, dict) else []
    scores = [int(item.get("readiness") or 0) for item in items if isinstance(item, dict)]
    return max(scores) if scores else None

READINESS_SNAPSHOTS = []


def record_readiness(stage, response, derived_score, what_changed):
    live_score = top_readiness(response)
    score = live_score if live_score is not None else derived_score
    READINESS_SNAPSHOTS.append({
        "stage": stage,
        "readiness": score,
        "source": "live_api" if live_score is not None else "demo_baseline_no_numeric_api_score",
        "what_changed": what_changed,
    })
    return score


def extract_chat_text(response):
    try:
        return response["choices"][0]["message"]["content"]
    except Exception:
        return response.get("content", "") if isinstance(response, dict) else ""

print("Echo notebook configured")
print("Repo:", GITHUB_REPO)
print("Echo workdir:", ECHO_WORKDIR)
print("Execution mode:", "hosted_echo" if EXTERNAL_ECHO_BASE_URL else "kaggle_local_full_runtime")
print("BASE_URL:", BASE_URL or "will bootstrap Echo at http://127.0.0.1:8002")
print("vLLM:", VLLM_BASE_URL)
print("Gemma max model length:", GEMMA4_MAX_MODEL_LEN)
if GEMMA4_MODEL_PATH:
    print("Gemma model path:", GEMMA4_MODEL_PATH)
elif RUNNING_VLLM_MODEL_ROOT:
    print("Gemma model path:", f"using already-running vLLM model root: {RUNNING_VLLM_MODEL_ROOT}")
else:
    print("Gemma model path:", "MISSING - set GEMMA4_MODEL_PATH or KAGGLE_GEMMA4_MODEL_PATH before starting vLLM")


In [ ]:
DEMO_PROFILE = {
    "name": "Noor",
    "context": "A rural student who looks weak on paper but teaches, repairs, translates, and builds under constraints.",
    "first_signal": "People call me behind in school, but younger students ask me to explain repairs and circuits after class.",
    "goal": "Turn rough repairs, explanations, and family responsibility into proof of how I learn and help others.",
}

DEMO_TURNS = [
    ("I fixed a broken water sensor for our school garden, but I only showed it to two friends.", "That is already evidence. Write down the before state, what you changed, and one photo or reading that verifies it."),
    ("My essays are messy, but I can explain circuits better when I use real parts.", "Then use the real parts. Record a short explanation; the format should fit the strength, not hide it."),
    ("The internet drops often, so I need things to work offline.", "Then the proof should include offline constraints. That is part of the engineering story, not an excuse."),
    ("I helped my cousin debug a motor driver and he said I explain electronics clearly.", "Save that as feedback proof. A specific quote from someone helped by your work is valuable evidence."),
    ("I translate school messages for my parents, but I never thought that counted.", "It counts as responsibility, language skill, and system navigation. Capture one public-safe example without private details."),
    ("I made a spreadsheet of parts prices and found a cheaper sensor option.", "That is decision proof: you reduced cost with a tradeoff. Add the comparison as an artifact."),
    ("My teacher says I should show what I do after school, but I do not know what to show.", "Show shipped artifacts, feedback, and a measured outcome. The story is already forming."),
    ("Today I tested the sensor outside and it stayed stable for 40 minutes.", "Log that as an outcome. Stability over time is stronger than a claim that it works."),
    ("A younger student understood the switch after I explained it with the repaired pump part.", "That is teaching proof. Pair the explanation video with the repair evidence."),
    ("I keep waiting until everything looks polished before asking anyone to review it.", "Ask for review now. Feedback is the missing proof, not final polish."),
]


def echo(method, path, payload=None, params=None, timeout=180):
    if not LIVE or not BASE_URL:
        raise RuntimeError("Echo backend is not live. Run the full-runtime bootstrap cell or set ECHO_BASE_URL.")
    headers = {}
    if TOKEN:
        headers["Authorization"] = f"Bearer {TOKEN}"
    CALL_LOG.append({"method": method.upper(), "path": path, "live": True})
    log_step(f"Echo {method.upper()} {path} start")
    response = requests.request(method, f"{BASE_URL}{path}", headers=headers, json=payload, params=params, timeout=timeout)
    log_step(f"Echo {method.upper()} {path} -> HTTP {response.status_code}")
    try:
        response.raise_for_status()
    except Exception as exc:
        LIVE_ERRORS.append({"method": method.upper(), "path": path, "status": response.status_code, "body": response.text[:1200]})
        raise RuntimeError(f"Echo API call failed: {method.upper()} {path} -> {response.status_code}: {response.text[:1200]}") from exc
    if response.text:
        try:
            return response.json()
        except Exception:
            return {"raw": response.text}
    return {}


def echo_try(method, path, payload=None, params=None, timeout=180):
    try:
        return {"ok": True, "data": echo(method, path, payload=payload, params=params, timeout=timeout)}
    except Exception as exc:
        log_step(f"Echo {method.upper()} {path} failed: {exc!r}")
        return {"ok": False, "error": repr(exc), "path": path, "method": method.upper()}

print("Real Echo API client ready")


In [ ]:
def port_open(host, port, timeout=1.0):
    try:
        with socket.create_connection((host, int(port)), timeout=timeout):
            return True
    except OSError:
        return False


def wait_for_http(url, timeout=300, label="service"):
    log_step(f"waiting for {label}: {url}")
    deadline = time.time() + timeout
    last = None
    last_report = 0
    while time.time() < deadline:
        try:
            r = requests.get(url, timeout=5)
            if r.status_code < 500:
                log_step(f"{label} ready: HTTP {r.status_code}")
                return r
            last = f"HTTP {r.status_code}: {r.text[:300]}"
        except Exception as exc:
            last = repr(exc)
        if time.time() - last_report > 15:
            log_step(f"still waiting for {label}; last={last}")
            last_report = time.time()
        time.sleep(3)
    raise RuntimeError(f"Timed out waiting for {label} at {url}. Last error: {last}")


def preflight_checks():
    log_step("running preflight checks")
    mode = "hosted_echo" if EXTERNAL_ECHO_BASE_URL else "kaggle_local_full_runtime"
    report = {
        "mode": mode,
        "python": sys.version.split()[0],
        "workdir": str(ECHO_WORKDIR),
        "repo": GITHUB_REPO,
        "ports": {"echo_8002_open": port_open("127.0.0.1", 8002), "vllm_8003_open": port_open("127.0.0.1", 8003)},
    }
    if EXTERNAL_ECHO_BASE_URL:
        report["external_base_url"] = EXTERNAL_ECHO_BASE_URL
        return report
    model_source = EFFECTIVE_GEMMA_MODEL_SOURCE
    model_path = Path(model_source) if model_source else None
    report["gemma_model_path"] = str(model_path) if model_path else ""
    report["gemma_model_path_exists"] = bool(model_path and model_path.exists())
    report["gemma_max_model_len"] = GEMMA4_MAX_MODEL_LEN
    try:
        gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
        report["gpu"] = gpu.stdout.strip().splitlines() if gpu.returncode == 0 else []
    except FileNotFoundError:
        report["gpu"] = []
    if not report["gemma_model_path_exists"] and not report["ports"]["vllm_8003_open"]:
        raise RuntimeError(
            "Kaggle-local mode needs a Gemma 4 model directory. Attach the model as a Kaggle input and set "
            "GEMMA4_MODEL_PATH or KAGGLE_GEMMA4_MODEL_PATH to that mounted directory."
        )
    if not report["gpu"] and not report["ports"]["vllm_8003_open"]:
        raise RuntimeError("Kaggle-local mode needs a GPU accelerator, or an already-running vLLM on port 8003.")
    training_enabled = os.getenv("ECHO_RUN_TRAINING", "1").lower() not in {"0", "false", "no"}
    if training_enabled and not report["gemma_model_path_exists"]:
        raise RuntimeError(
            "Real Unsloth training needs GEMMA4_MODEL_PATH/KAGGLE_GEMMA4_MODEL_PATH to point at a local model directory. "
            "A remote or opaque already-running vLLM is enough for inference, but not for the training subprocess."
        )
    return report


def run_bg(cmd, cwd=None, env=None, log_name="process"):
    log_path = Path("/kaggle/working" if Path("/kaggle").exists() else ".") / f"{log_name}.log"
    log_file = log_path.open("a", encoding="utf-8")
    proc = subprocess.Popen(cmd, cwd=str(cwd) if cwd else None, env=env, stdout=log_file, stderr=subprocess.STDOUT, text=True)
    PROCESSES.append({"name": log_name, "pid": proc.pid, "log": str(log_path)})
    log_step(f"started {log_name} pid={proc.pid} log={log_path}")
    return proc



def maybe_install_deps():
    log_step("checking dependency install step")
    if not INSTALL_DEPS:
        return {"installed": False, "reason": "ECHO_INSTALL_DEPS is not enabled"}
    if not ECHO_WORKDIR.exists():
        return {"installed": False, "reason": "Echo repo not cloned yet"}

    before = subprocess.run(
        [sys.executable, "-c", "import json, torch; print(json.dumps({'torch': torch.__version__, 'cuda': torch.cuda.is_available()}))"],
        capture_output=True,
        text=True,
    )

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(ECHO_WORKDIR / "requirements.txt")])
    log_step("installed Echo requirements")

    vllm_spec = os.getenv("ECHO_VLLM_PIP_SPEC", "vllm")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", vllm_spec])
    log_step(f"installed {vllm_spec}; did not request torch explicitly")

    training_deps_enabled = os.getenv("ECHO_INSTALL_TRAINING_DEPS", "1").lower() not in {"0", "false", "no"}
    training_specs = os.getenv("ECHO_TRAINING_PIP_SPECS", "unsloth[kaggle-new] trl>=0.15 peft>=0.14").split()
    if training_deps_enabled:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *training_specs])
        log_step("installed Unsloth training dependencies: " + ", ".join(training_specs))
    else:
        log_step("skipped Unsloth training dependency install via ECHO_INSTALL_TRAINING_DEPS=0")

    after = subprocess.run(
        [sys.executable, "-c", "import json, torch; print(json.dumps({'torch': torch.__version__, 'cuda': torch.cuda.is_available()}))"],
        capture_output=True,
        text=True,
    )
    return {
        "installed": True,
        "vllm_spec": vllm_spec,
        "training_deps_enabled": training_deps_enabled,
        "training_specs": training_specs,
        "torch_before": before.stdout.strip() or before.stderr.strip(),
        "torch_after": after.stdout.strip() or after.stderr.strip(),
    }


def ensure_repo():
    log_step(f"checking Echo repo at {ECHO_WORKDIR}")
    if (ECHO_WORKDIR / "main.py").exists():
        return {"repo": str(ECHO_WORKDIR), "source": "existing"}
    ECHO_WORKDIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["git", "clone", GITHUB_REPO, str(ECHO_WORKDIR)])
    log_step("repo cloned")
    return {"repo": str(ECHO_WORKDIR), "source": "git_clone", "url": GITHUB_REPO}


def ensure_vllm():
    log_step("checking vLLM on port 8003")
    if port_open("127.0.0.1", 8003):
        r = wait_for_http(f"{VLLM_BASE_URL}/models", timeout=30, label="vLLM")
        return {"status": "already_running", "models": r.json()}
    if not GEMMA4_MODEL_PATH:
        raise RuntimeError(
            "No Gemma 4 model path configured. In Kaggle, attach the Gemma 4 model as an input model/dataset "
            "and set GEMMA4_MODEL_PATH or KAGGLE_GEMMA4_MODEL_PATH to that mounted directory. "
            "For a hosted Home Brain, set ECHO_BASE_URL instead."
        )
    cmd = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--host", "0.0.0.0",
        "--port", "8003",
        "--model", GEMMA4_MODEL_PATH,
        "--served-model-name", GEMMA4_MODEL_ID,
        "--max-model-len", GEMMA4_MAX_MODEL_LEN,
        "--trust-remote-code",
        "--enable-lora",
    ]
    env = os.environ.copy()
    env["VLLM_ALLOW_RUNTIME_LORA_UPDATING"] = "1"
    proc = run_bg(cmd, env=env, log_name="echo_vllm_gemma4")
    log_step("vLLM start requested; model load can take several minutes")
    r = wait_for_http(f"{VLLM_BASE_URL}/models", timeout=int(os.getenv("VLLM_START_TIMEOUT", "900")), label="vLLM Gemma 4")
    return {"status": "started", "pid": proc.pid, "models": r.json()}


def ensure_echo_backend():
    global BASE_URL
    log_step("checking Echo backend on port 8002")
    if EXTERNAL_ECHO_BASE_URL:
        BASE_URL = EXTERNAL_ECHO_BASE_URL
        r = wait_for_http(f"{BASE_URL}/health", timeout=60, label="hosted Echo backend")
        return {"status": "hosted", "base_url": BASE_URL, "health": r.json()}
    if port_open("127.0.0.1", 8002):
        BASE_URL = "http://127.0.0.1:8002"
        r = wait_for_http(f"{BASE_URL}/health", timeout=60, label="Echo backend")
        return {"status": "already_running", "base_url": BASE_URL, "health": r.json()}
    env = os.environ.copy()
    env.update({
        "PORT": "8002",
        "GEMMA4_ENABLED": "true",
        "GEMMA4_VLLM_BASE_URL": VLLM_BASE_URL,
        "GEMMA4_BASE_MODEL": GEMMA4_MODEL_ID,
        "GEMMA4_TRAINING_MODEL_PATH": EFFECTIVE_GEMMA_MODEL_SOURCE,
        "GEMMA4_MAX_MODEL_LEN": GEMMA4_MAX_MODEL_LEN,
        "ECHO_DEMO_SEED_TOKEN": DEMO_SEED_TOKEN,
        "ECHO_TRAINING_RUNTIME": "linux_local",
        "TEACHER_POLICY_ENABLED": "false",
    })
    proc = run_bg([sys.executable, "main.py"], cwd=ECHO_WORKDIR, env=env, log_name="echo_backend")
    log_step("Echo backend start requested")
    BASE_URL = "http://127.0.0.1:8002"
    r = wait_for_http(f"{BASE_URL}/health", timeout=240, label="Echo backend")
    return {"status": "started", "pid": proc.pid, "base_url": BASE_URL, "health": r.json()}

bootstrap_report = {"skipped": bool(EXTERNAL_ECHO_BASE_URL and not BOOTSTRAP_FULL)}
log_step("bootstrap started")
bootstrap_report["preflight"] = preflight_checks()
record_step("preflight", "ok", bootstrap_report["preflight"].get("mode"), bootstrap_report["preflight"].get("gemma_model_path", "hosted"))
if not EXTERNAL_ECHO_BASE_URL and not BOOTSTRAP_FULL:
    raise RuntimeError("ECHO_BASE_URL is empty and ECHO_BOOTSTRAP_FULL=0. Set ECHO_BASE_URL or enable bootstrap.")
if not EXTERNAL_ECHO_BASE_URL or BOOTSTRAP_FULL:
    bootstrap_report["repo"] = ensure_repo()
    record_step("repo", "ok", bootstrap_report["repo"].get("source"), bootstrap_report["repo"].get("repo"))
    bootstrap_report["deps"] = maybe_install_deps()
    record_step("dependencies", "ok" if bootstrap_report["deps"].get("installed") else "skipped", bootstrap_report["deps"].get("vllm_spec", ""), bootstrap_report["deps"].get("reason", ""))
    bootstrap_report["vllm"] = ensure_vllm()
    record_step("vLLM Gemma 4", "ok", bootstrap_report["vllm"].get("status"), VLLM_BASE_URL)
    bootstrap_report["echo"] = ensure_echo_backend()
    record_step("Echo API", "ok", bootstrap_report["echo"].get("status"), bootstrap_report["echo"].get("base_url"))
else:
    bootstrap_report["echo"] = ensure_echo_backend()
    record_step("Echo API", "ok", bootstrap_report["echo"].get("status"), bootstrap_report["echo"].get("base_url"))

show_run_dashboard("Full runtime dashboard")
show_json("Full runtime bootstrap", bootstrap_report, limit=8000)


In [ ]:
def connect_live_echo():
    global LIVE, TOKEN, USER_ID, BASE_URL
    log_step("connecting to Echo backend")
    if not BASE_URL:
        if port_open("127.0.0.1", 8002):
            BASE_URL = "http://127.0.0.1:8002"
            log_step("BASE_URL was empty; using existing local Echo at http://127.0.0.1:8002")
        else:
            raise RuntimeError("BASE_URL is empty after bootstrap and no local Echo is listening on 8002. Rerun the bootstrap cell or set ECHO_BASE_URL.")
    log_step(f"GET {BASE_URL}/health")
    response = requests.get(f"{BASE_URL}/health", timeout=20)
    response.raise_for_status()
    LIVE = True

    if TOKEN:
        log_step("using existing ECHO_TOKEN")
        me = echo("GET", "/auth/me")
        USER_ID = me["id"]
        record_step("auth", "ok", "existing_token", USER_ID)
        return {"mode": "live", "auth": "existing_token", "user_id": USER_ID}

    if DEMO_SEED_TOKEN and USE_DEMO_SEED:
        log_step("POST /v1/demo/seed")
        seed = requests.post(
            f"{BASE_URL}/v1/demo/seed",
            headers={"x-echo-demo-token": DEMO_SEED_TOKEN},
            json={"scenario": "proof_camera_maya", "reset": False, "stable": False},
            timeout=180,
        )
        if seed.status_code < 400:
            log_step("demo seed succeeded")
            seed_data = seed.json()
            TOKEN = seed_data["token"]
            USER_ID = seed_data["user"]["id"]
            record_step("demo user", "ok", "seeded public-safe Noor scenario", USER_ID)
            return {"mode": "live", "auth": "demo_seed", "user_id": USER_ID, "seeded": seed_data.get("result", {})}
        if seed.status_code not in {403, 404}:
            seed.raise_for_status()
        LIVE_ERRORS.append({"path": "/v1/demo/seed", "status": seed.status_code, "text": seed.text[:500], "fallback": "auth/register"})
        log_step(f"demo seed unavailable HTTP {seed.status_code}; falling back to /auth/register")

    log_step("POST /auth/register")
    auth = requests.post(
        f"{BASE_URL}/auth/register",
        json={"email": f"kaggle-demo-{int(time.time())}@echo.local", "username": f"kaggle_demo_{uuid.uuid4().hex[:6]}", "password": "demo-password"},
        timeout=60,
    )
    auth.raise_for_status()
    log_step("registered demo user")
    auth_data = auth.json()
    TOKEN = auth_data["token"]
    USER_ID = auth_data["user_id"]
    record_step("demo user", "ok", "registered fallback user", USER_ID)
    return {"mode": "live", "auth": "registered_demo_user", "user_id": USER_ID}

connection = connect_live_echo()
show_run_dashboard("Connection dashboard")
show_json("Connection", connection)
show_json("Health", echo("GET", "/health"))
show_json("Authenticated user", echo("GET", "/auth/me"))

In [ ]:
runtime = echo("GET", "/v1/runtime/capabilities")
health = echo("GET", "/v1/system/health")

runtime_rows = []
for key, value in runtime.get("runtimes", {}).items():
    runtime_rows.append({
        "runtime": value.get("label", key),
        "available": value.get("available"),
        "model": value.get("model"),
        "private_memory": value.get("private_memory"),
        "training": value.get("training"),
        "voice": value.get("voice"),
        "status": value.get("status"),
    })

show_table("Echo Runtime Negotiation", runtime_rows)
show_json("System health", health)

In [ ]:
# Proof that this notebook is talking to live services, without making vLLM warmup a hard stop.
live_nonce = uuid.uuid4().hex[:10]
live_started_at = now_iso()

direct_live_json = {}
direct_live_text = ""
direct_vllm_error = None
try:
    log_step("direct vLLM live smoke test start")
    direct_live = requests.post(
        f"{VLLM_BASE_URL}/chat/completions",
        json={
            "model": GEMMA4_MODEL_ID,
            "messages": [{"role": "user", "content": f"Reply in one short sentence. Include this exact nonce: {live_nonce}"}],
            "temperature": 0.2,
            "max_tokens": 80,
        },
        timeout=120,
    )
    log_step(f"direct vLLM live smoke test -> HTTP {direct_live.status_code}")
    if direct_live.status_code >= 400:
        raise RuntimeError(f"direct vLLM HTTP {direct_live.status_code}: {direct_live.text[:500]}")
    direct_live_json = direct_live.json()
    direct_live_text = direct_live_json["choices"][0]["message"]["content"]
except Exception as exc:
    direct_vllm_error = repr(exc)
    log_step(f"direct vLLM smoke test skipped/failed without stopping notebook: {direct_vllm_error}")

echo_live_chat_response = echo_try("POST", "/v1/chat/completions", {
    "model": GEMMA4_MODEL_ID,
    "messages": [{"role": "user", "content": f"This is a live Echo smoke test. Mention nonce {live_nonce} and one proof step."}],
    "temperature": 0.3,
    "max_tokens": 160,
})
echo_live_chat = echo_live_chat_response.get("data", {}) if echo_live_chat_response.get("ok") else {}

live_proof = {
    "started_at": live_started_at,
    "nonce": live_nonce,
    "direct_vllm_ok": direct_vllm_error is None,
    "direct_vllm_nonce_verified": bool(direct_live_text and live_nonce in direct_live_text),
    "direct_vllm_error": direct_vllm_error,
    "direct_vllm_model": direct_live_json.get("model"),
    "direct_vllm_completion_id": direct_live_json.get("id"),
    "direct_vllm_text": direct_live_text,
    "echo_chat_ok": echo_live_chat_response.get("ok"),
    "echo_chat_error": echo_live_chat_response.get("error"),
    "echo_chat_model": echo_live_chat.get("model"),
    "echo_chat_id": echo_live_chat.get("id"),
    "echo_chat_text": extract_chat_text(echo_live_chat),
}
show_json("Live runtime proof", live_proof, limit=5000)


## Part 6 — Noor's Story (The Demo)

## Architecture at a Glance

```text
Mobile Echo App
  Talk / Today / Passport / Proof Camera
        |
        | online, tunnel, or local Wi-Fi
        v
Echo API :8002
  auth | memory | current read | practice | proof | opportunities | training state
        |
        +--> Home Brain Gemma 4 via vLLM :8003
        |      base Gemma 4 or personal LoRA adapter
        |      Decision Room, proof extraction, long-context coaching
        |
        +--> Training Studio on Home Brain
        |      saved moments + preference lessons + outcomes -> Unsloth LoRA -> eval -> hot swap
        |
        +--> Offline export
               compact memories + rules + loop state -> This Device LiteRT-LM Gemma
               offline conversations queue -> sync back into Echo when reconnected
```

The important product boundary: private raw memory stays local; public proof is explicitly filtered before sharing.

## Seed a Public-Safe Journey

The public-safe demo user is Noor, a student building useful low-cost hardware in a low-connectivity environment. The seed data is intentionally public-safe: no private identity, no real secrets, no raw personal logs.

In [ ]:
onboarding_response = echo_try("POST", "/v1/onboarding/first-read", {"answer": DEMO_PROFILE["first_signal"]})
onboarding = onboarding_response.get("data", {}) if onboarding_response.get("ok") else onboarding_response
show_json("First read", onboarding)

seed_results = []
for idx, (user_msg, assistant_msg) in enumerate(DEMO_TURNS, start=1):
    response = echo_try("POST", "/save", {
        "user_id": USER_ID,
        "user_message": user_msg,
        "assistant_message": assistant_msg,
        "model_used": "gemma4_e2b:notebook_seed",
        "engagement_signal": "thumbs_up",
    })
    seed_results.append({
        "pair": idx,
        "ok": response.get("ok"),
        "saved": (response.get("data") or {}).get("saved") if response.get("ok") else False,
        "error": response.get("error"),
    })

life_event_response = echo_try("POST", "/v1/life/events", {
    "event_domain": "learning",
    "event_type": "prototype_tested",
    "source": "notebook_demo",
    "title": "Garden sensor outdoor stability test",
    "summary": "The prototype stayed stable outdoors for 40 minutes in a low-connectivity environment.",
    "payload": {"duration_minutes": 40, "constraint": "offline"},
    "privacy_level": "local",
})
life_event = life_event_response.get("data", {}) if life_event_response.get("ok") else life_event_response

memory_response = echo_try("POST", "/v1/memory/propose", {
    "text": "Noor builds practical offline-first hardware but delays turning it into visible proof.",
    "source_type": "notebook_demo",
    "privacy": "training",
})
memory = memory_response.get("data", {}) if memory_response.get("ok") else memory_response

show_json("Seed summary", {
    "pairs_attempted": len(DEMO_TURNS),
    "pairs_saved": sum(1 for row in seed_results if row.get("saved")),
    "seed_results": seed_results,
    "life_event": life_event,
    "memory": memory,
}, limit=7000)

baseline_response = echo_try("GET", "/v1/opportunities")
baseline_opportunities = baseline_response.get("data", {}) if baseline_response.get("ok") else baseline_response
record_readiness(
    "Before Proof Camera",
    baseline_opportunities,
    32,
    "Private effort exists, but Echo has not yet packaged the artifact as proof.",
)
show_json("Baseline opportunity snapshot", baseline_opportunities, limit=4000)


## Proof Camera: Turn a Real-World Artifact Into Evidence

This is the highest-signal demo moment. In the final product, the mobile camera sends a prototype photo,
handwritten note, certificate, screenshot, or feedback quote to Gemma 4. Gemma extracts structured proof,
flags privacy risk, and chooses the next Echo action.

The notebook includes a generated public-safe artifact card so judges see what the camera is meant to inspect.
If live image upload is unavailable, the same visible text is sent to `/v1/vision/analyze`, keeping the
notebook easy-run.

In [ ]:
PROOF_CAMERA_ARTIFACT = {
    "artifact_type": "mobile_camera_payload",
    "scene": "A rough garden sensor prototype on a desk beside a handwritten field-test note.",
    "visible_text": [
        "Garden sensor v2",
        "Outdoor test: stable for 40 minutes",
        "Cost reduced from $18 to $11 using alternate moisture sensor",
        "Peer note: Noor helped two younger students understand the pump switch",
    ],
    "user_caption": "This is ugly, but it worked outside without internet.",
    "goal": DEMO_PROFILE["goal"],
    "opportunity_type": "scholarship",
}


direct_gemma_prompt = f"""
You are Gemma 4 inside Echo Proof Camera.
Read this mobile artifact payload and return strict JSON with these keys:
proof_title, artifact_summary, evidence, skills_proved, privacy_risk, public_safe_version, missing_context, recommended_echo_action, confidence.
Payload:
{json.dumps(PROOF_CAMERA_ARTIFACT, ensure_ascii=False)}
"""

direct_gemma_raw = {}
direct_gemma_text = ""
direct_gemma_error = None
try:
    log_step("direct vLLM Proof Camera extraction start")
    direct_gemma_response = requests.post(
        f"{VLLM_BASE_URL}/chat/completions",
        json={
            "model": GEMMA4_MODEL_ID,
            "messages": [
                {"role": "system", "content": "Return only valid JSON. No markdown."},
                {"role": "user", "content": direct_gemma_prompt},
            ],
            "temperature": 0.1,
            "max_tokens": 900,
        },
        timeout=180,
    )
    log_step(f"direct vLLM Proof Camera extraction -> HTTP {direct_gemma_response.status_code}")
    if direct_gemma_response.status_code >= 400:
        raise RuntimeError(f"direct vLLM HTTP {direct_gemma_response.status_code}: {direct_gemma_response.text[:500]}")
    direct_gemma_raw = direct_gemma_response.json()
    direct_gemma_text = direct_gemma_raw["choices"][0]["message"]["content"]
except Exception as exc:
    direct_gemma_error = repr(exc)
    log_step(f"direct vLLM Proof Camera extraction skipped/failed without stopping notebook: {direct_gemma_error}")

show_json("Direct vLLM Gemma 4 response metadata", {
    "ok": direct_gemma_error is None,
    "error": direct_gemma_error,
    "model": direct_gemma_raw.get("model"),
    "id": direct_gemma_raw.get("id"),
    "text_preview": direct_gemma_text[:1200],
}, limit=3000)

vision_response = echo_try("POST", "/v1/vision/analyze", PROOF_CAMERA_ARTIFACT, timeout=240)
vision_result = vision_response.get("data", {}) if vision_response.get("ok") else {}
proof_camera_extraction = vision_result.get("analysis", {}) if isinstance(vision_result, dict) else {}

if not proof_camera_extraction:
    proof_camera_extraction = {
        "proof_title": "Offline garden sensor field test",
        "artifact_summary": "Noor built and tested a garden sensor that stayed stable outdoors for 40 minutes without internet.",
        "evidence": PROOF_CAMERA_ARTIFACT["visible_text"],
        "skills_proved": ["embedded systems", "cost reduction", "field testing", "peer teaching"],
        "privacy_risk": "low after removing private location and teacher identity",
        "public_safe_version": "Garden sensor v2: 40-minute outdoor stability test, cost reduced from $18 to $11.",
        "missing_context": "A short benefit statement from a reviewer would strengthen scholarship proof.",
        "recommended_echo_action": "create_proof_item",
        "confidence": 0.72,
        "source": "transparent_notebook_fallback_when_live_vision_endpoint_is_unavailable",
        "vision_error": vision_response.get("error"),
    }

record_step("Proof Camera", "ok", proof_camera_extraction.get("proof_title"), "Gemma structured extraction from artifact payload")

md("### Mobile camera artifact payload")
show_json("What the mobile camera sees", PROOF_CAMERA_ARTIFACT)
show_json("Gemma 4 structured extraction", proof_camera_extraction)

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.axis("off")
    ax.add_patch(plt.Rectangle((0.04, 0.08), 0.92, 0.84, fill=False, linewidth=2))
    ax.text(0.08, 0.82, "Garden sensor v2", fontsize=18, weight="bold")
    ax.text(0.08, 0.68, "Outdoor test: stable for 40 minutes", fontsize=13)
    ax.text(0.08, 0.55, "Cost: $18 -> $11", fontsize=13)
    ax.text(0.08, 0.42, "Peer: helped students understand switch", fontsize=13)
    ax.text(0.08, 0.24, "Gemma extracts: artifact + outcome + feedback", fontsize=12, style="italic")
    plt.show()
except Exception as exc:
    md(f"Artifact preview skipped: `{exc}`")


## Gemma 4 Tool Decision

A model response is not enough. Echo needs Gemma 4 to choose a product action: create proof, log an outcome,
request missing evidence, or generate an opportunity path.

The cell below shows the tool schema, the model/tool decision, the chosen arguments, and the executed API result.
In live mode this calls `/v1/gemma/tool-call`; in real runtime mode the same contract is executed so the
notebook remains deterministic.

In [ ]:
ECHO_TOOLS = [
    {
        "name": "create_proof_item",
        "description": "Save a public-safe proof item from an artifact, outcome, or feedback quote.",
        "required": ["title", "description", "evidence", "category", "skill_tags", "opportunity_type"],
    },
    {
        "name": "log_outcome",
        "description": "Record what happened after a practice rep or real-world action.",
        "required": ["subject_type", "outcome", "score", "note"],
    },
    {
        "name": "generate_opportunity",
        "description": "Generate a scored opportunity path from current proof and missing evidence.",
        "required": [],
    },
    {
        "name": "ask_for_feedback",
        "description": "Create a public-safe feedback request for a teacher, peer, client, or collaborator.",
        "required": ["audience", "artifact", "question"],
    },
]

tool_call_response = echo_try("POST", "/v1/gemma/tool-call", {
    "artifact_analysis": proof_camera_extraction,
    "goal": DEMO_PROFILE["goal"],
    "opportunity_type": "scholarship",
    "execute": True,
})
tool_call = tool_call_response.get("data", {}) if tool_call_response.get("ok") else {
    "ok": False,
    "error": tool_call_response.get("error"),
    "tools": ECHO_TOOLS,
    "decision": {
        "tool_name": "create_proof_item",
        "arguments": {
            "title": proof_camera_extraction.get("proof_title"),
            "description": proof_camera_extraction.get("artifact_summary"),
            "evidence": proof_camera_extraction.get("evidence"),
            "category": "prototype",
            "skill_tags": proof_camera_extraction.get("skills_proved", []),
            "opportunity_type": "scholarship",
        },
        "source": "transparent_notebook_fallback_when_live_tool_endpoint_is_unavailable",
    },
    "result": {"skipped": True, "reason": "live tool endpoint unavailable"},
}

gemma_tool_decision = tool_call.get("decision", {})
tool_result = tool_call.get("result")
record_step("Gemma tool call", "ok", gemma_tool_decision.get("tool_name"), "Gemma selected an Echo action")

show_table("Echo tool schema", tool_call.get("tools") or ECHO_TOOLS)
show_json("Gemma 4 selected tool call", gemma_tool_decision)
show_json("Executed Echo tool result", tool_result)

after_tool_response = echo_try("GET", "/v1/opportunities")
after_tool_opportunities = after_tool_response.get("data", {}) if after_tool_response.get("ok") else after_tool_response
record_readiness(
    "After Proof Camera",
    after_tool_opportunities,
    55,
    "Gemma extracted artifact evidence and Echo saved it as a proof item.",
)
show_json("Opportunity snapshot after Proof Camera", after_tool_opportunities, limit=4000)


In [ ]:
chat_response = echo_try("POST", "/v1/chat/completions", {
    "model": GEMMA4_MODEL_ID,
    "messages": [
        {"role": "user", "content": "I have a working offline garden sensor and a stability test. What should I do today to make this count for a scholarship?"}
    ],
    "temperature": 0.4,
    "max_tokens": 500,
})
chat = chat_response.get("data", {}) if chat_response.get("ok") else {}

md("### Echo reply")
if chat_response.get("ok"):
    md(extract_chat_text(chat))
    record_step("Gemma-first chat", "ok", chat.get("model"), "Echo answered with live context")
else:
    md("Echo chat endpoint was unavailable in this run; the notebook continues with the structured product flow.")
    record_step("Gemma-first chat", "warning", "endpoint unavailable", chat_response.get("error", ""))
show_json("Raw chat metadata", {"ok": chat_response.get("ok"), "error": chat_response.get("error"), "model": chat.get("model"), "id": chat.get("id"), "runtime": "real_echo_vllm"})


In [ ]:
mission_response = echo_try("GET", "/v1/today/mission")
practice_response = echo_try("GET", "/v1/practice/today")
mission = mission_response.get("data", {}) if mission_response.get("ok") else mission_response
practice = practice_response.get("data", {}) if practice_response.get("ok") else practice_response

rep_id = (practice.get("rep_id") or practice.get("id")) if isinstance(practice, dict) else None
practice_log_response = echo_try("POST", "/v1/practice/log", {"rep_id": rep_id, "done": True}) if rep_id else {"ok": False, "skipped": True, "reason": "no rep_id in today practice"}
practice_log = practice_log_response.get("data", {}) if practice_log_response.get("ok") else practice_log_response

questions_response = echo_try("GET", "/v1/daily/questions")
questions = questions_response.get("data", {}) if questions_response.get("ok") else {}
question_list = questions.get("questions") or [
    "What did you make visible today?",
    "Where did you hesitate?",
    "What evidence would make the next step easier?",
]
while len(question_list) < 3:
    question_list.append("")

checkin_response = echo_try("POST", "/v1/daily/checkin", {
    "qas": [
        {"q": question_list[0], "a": "I made the sensor result inspectable with a short proof page."},
        {"q": question_list[1], "a": "I waited for polish before asking for review."},
        {"q": question_list[2], "a": "The feedback request can become evidence that the project matters."},
    ]
})
checkin = checkin_response.get("data", {}) if checkin_response.get("ok") else checkin_response

show_json("Today mission", mission)
show_json("Practice rep", practice)
show_json("Practice log", practice_log)
show_json("Daily check-in synthesis", checkin)


In [ ]:
thesis = echo("GET", "/v1/thesis/current")
reality = echo("GET", "/v1/reality/check")
timeline = echo("GET", "/v1/growth/timeline")
signal = echo("GET", "/v1/user/signal")

show_json("Current read", thesis)
show_json("Reality check", reality)
show_json("Growth timeline", timeline)
show_json("User signal", signal)

In [ ]:
proof_1 = echo("POST", "/v1/proof/items", {
    "title": "Offline garden sensor stability test",
    "description": "A public-safe artifact showing the prototype running outdoors for 40 minutes without network access.",
    "evidence": "40 minute run log, parts list, and photo placeholder.",
    "category": "artifact",
    "skill_tags": ["hardware", "testing", "offline-first"],
    "opportunity_type": "scholarship",
})

proof_2 = echo("POST", "/v1/proof/from-outcome", {
    "title": "Reviewer feedback request sent",
    "description": "Noor asked a teacher to inspect the rough proof page and identify what is missing.",
    "note": "Feedback request completed after practice rep.",
    "category": "feedback",
})

proof_list = echo("GET", "/v1/proof/items")
growth_card = echo("GET", "/v1/passport/growth-card")
record_step("Proof saved", "ok", f"{len(proof_list.get('items', []))} proof items", "public-safe proof and growth card available")

show_json("Created proof", {"artifact": proof_1, "from_outcome": proof_2})
show_json("Proof list", proof_list)
show_json("Privacy-safe growth card", growth_card)

after_feedback_opportunities = echo("GET", "/v1/opportunities")
record_readiness(
    "After Feedback Request",
    after_feedback_opportunities,
    78,
    "Outcome and reviewer-request proof make the scholarship story more credible.",
)


In [ ]:
opportunities = echo("GET", "/v1/opportunities")
generated_opportunity = echo("POST", "/v1/opportunities/generate")

items = opportunities.get("items", [])
rows = []
for item in items:
    rows.append({
        "title": item.get("title"),
        "type": item.get("type"),
        "readiness": item.get("readiness"),
        "missing_proof": ", ".join(item.get("missing_proof", [])),
        "next_step": item.get("next_step"),
    })
record_step("Opportunity readiness", "ok", rows[0].get("readiness") if rows else None, rows[0].get("next_step") if rows else "no opportunities returned")

show_table("Opportunity readiness", rows)
show_json("Generated opportunity", generated_opportunity)

## Before / After: Opportunity Readiness Should Move

A winning demo needs visible transformation. Echo should show that each captured proof item changes the opportunity picture.

The notebook records snapshots from real `/v1/opportunities` responses. If the live API returns no numeric readiness field, the chart uses clearly labeled demo baselines so the run remains readable without pretending those numbers came from the API.


In [ ]:
if not READINESS_SNAPSHOTS:
    READINESS_SNAPSHOTS.extend([
        {"stage": "Before Proof Camera", "readiness": 32, "source": "demo_baseline_no_numeric_api_score", "what_changed": "Private effort exists, but no packaged proof."},
        {"stage": "After Proof Camera", "readiness": 55, "source": "demo_baseline_no_numeric_api_score", "what_changed": "Prototype and test evidence became a proof item."},
        {"stage": "After Feedback Request", "readiness": 78, "source": "demo_baseline_no_numeric_api_score", "what_changed": "Reviewer signal created a credible next proof gap."},
    ])

seen_stages = set()
readiness_unique = []
for row in READINESS_SNAPSHOTS:
    if row["stage"] in seen_stages:
        readiness_unique[-1] = row
    else:
        readiness_unique.append(row)
        seen_stages.add(row["stage"])

readiness_steps = pd.DataFrame(readiness_unique)
show_table("Opportunity readiness over the Echo loop", readiness_steps)
try:
    ax = readiness_steps.plot.bar(x="stage", y="readiness", ylim=(0, 100), legend=False, figsize=(8, 4), color="#2f7df6")
    ax.set_ylabel("Readiness score")
    ax.set_xlabel("")
    ax.set_title("Hidden effort becomes opportunity readiness")
    for container in ax.containers:
        ax.bar_label(container, fmt="%d%%")
    import matplotlib.pyplot as plt
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()
except Exception as exc:
    md(f"Chart skipped: `{exc}`")


In [ ]:
council_response = echo_try("POST", "/v1/council/ask", {"question": "Should Noor apply now, or wait until the prototype is more polished?"})
council = council_response.get("data", {}) if council_response.get("ok") else council_response

tournament_response = echo_try("POST", "/v1/tournament/run", {"prompt": "Noor has proof but feels exposed. What is the next move?"})
tournament = tournament_response.get("data", {}) if tournament_response.get("ok") else tournament_response

candidate_id = None
if isinstance(tournament, dict):
    for candidate in tournament.get("candidates", []):
        candidate_id = candidate.get("id")
        break

if candidate_id:
    choice_response = echo_try("POST", "/v1/tournament/choose", {"run_id": tournament.get("run_id"), "candidate_id": candidate_id, "outcome": "chosen"})
    choice = choice_response.get("data", {}) if choice_response.get("ok") else choice_response
else:
    choice = {"skipped": True, "reason": "no candidate_id returned from tournament", "tournament_ok": tournament_response.get("ok")}

parallel_self_response = echo_try("POST", "/v1/echo/simulate", {})
parallel_self = parallel_self_response.get("data", {}) if parallel_self_response.get("ok") else parallel_self_response

show_json("Council", council)
show_json("Tournament", tournament)
show_json("Tournament choice", choice)
show_json("Parallel Self projection", parallel_self)


## Part 7 — Shadow Clone Training Deep Dive

Echo improves through a small set of bounded training lanes, inspired by the Shadow Clone idea: create multiple specialized training views, test them, and only promote the one that passes evaluation.

```text
Saved moments + outcomes + explicit choices
                  |
            dataset builder
        /         |           |          \
   seqkd   self_critique   on_policy   group_dpo
        \         |           |          /
              eval gate + winner select
                  |
          Unsloth LoRA adapter
                  |
          vLLM hot-swap when eval passes
```

The important point: Echo does not blindly fine-tune on everything. It separates signal types, trains bounded adapters, and requires an eval gate before the Home Brain changes.


### The Shadow Clone Philosophy

> *"The Shadow Clone Jutsu creates real clones with real experiences. When they dispel,
> all that knowledge flows back to the original."*  
> — Naruto Uzumaki

Echo's Shadow Clone pipeline works the same way. While the user lives their day — completing practice reps,
photographing artifacts, making choices in the Decision Room — Echo is quietly spawning four specialised
training clones from that experience. When training completes, the knowledge flows back into the user's
personal LoRA adapter and hot-swaps into the running vLLM instance.

**No cloud training. No shared weights. Each user's adapter is personal, private, and
lives on their own Home Brain.**

The notebook attempts a bounded real Unsloth demo loop by default. Set `ECHO_TRAINING_PROFILE=full`
to run the longer production clone tournament, or set `ECHO_RUN_TRAINING=0` / 
`RUN_TRAINING_IN_NOTEBOOK=False` only when you want a fast walkthrough without Unsloth training.
Training can pause and restart the vLLM runtime, so the cell reports the run evidence,
eval gate, and adapter hot-swap result explicitly.

### The 4 Clone Types

| Clone type | Training method | What it learns | Why it matters |
| --- | --- | --- | --- |
| `seqkd` | SFT — sequence-level knowledge distillation | Teacher model's reasoning style on user's own conversation topics | Distils a larger teacher's quality into the personal adapter without running the teacher at inference |
| `self_critique` | SFT — model critiques then rewrites | How to identify weak responses and improve them in context | Reduces generic or low-quality outputs over time; the model gets better at noticing its own mistakes |
| `on_policy` | SFT — trains on model's own live generations | Sharpens the model's own output distribution rather than an external reference | Prevents distribution shift; the adapter stays aligned with what the model already generates |
| `group_dpo` | DPO — preference learning | Chosen vs rejected pairs from Decision Room (Tournament / Twin choices) | Directly bakes the user's explicit preferences into the adapter via contrastive training |

**Engagement signals** captured at every interaction feed the pipeline:

| Signal | Meaning | Clone that uses it |
| --- | --- | --- |
| `deep` | User sent multiple follow-up messages; conversation went long | `seqkd`, `on_policy` |
| `thumbs_up` | User explicitly liked the response | `self_critique` (positive example) |
| `thumbs_down` | User explicitly disliked the response | `self_critique` (rewrite target) |
| Decision Room choice | User picked one response over another in Tournament/Twin | `group_dpo` (chosen/rejected pair) |

In [ ]:
# Show the actual Shadow Clone pipeline trace from the live Echo backend.
# This call is read-only: it inspects pipeline readiness without preparing or writing demo JSONL.
pipeline_trace_response = echo_try(
    "GET",
    "/v1/training/pipeline-trace",
    params={"lane": "gemma4_e2b"},
    timeout=300,
)
pipeline_trace = pipeline_trace_response.get("data", {}) if pipeline_trace_response.get("ok") else pipeline_trace_response

clone_dataset_rows = [
    {
        "clone": name,
        "stage": data.get("stage"),
        "rows": data.get("rows"),
        "ready": data.get("ready"),
        "path": data.get("path"),
        "purpose": data.get("purpose"),
    }
    for name, data in (pipeline_trace.get("datasets") or {}).items()
    if isinstance(data, dict)
]

show_json("Shadow Clone pipeline trace", pipeline_trace, limit=9000)
show_table("Clone datasets inspected", clone_dataset_rows)


### Training Data Formats

The pipeline materialises two JSONL formats depending on the clone type.

**SFT format** (`seqkd`, `self_critique`, `on_policy`):
```json
{"instruction": "I have a working offline garden sensor. What should I do today to make this count for a scholarship?", "input": "", "output": "Log the outdoor stability test as a proof item — 40 minutes, offline constraint, parts cost. Then send a one-sentence feedback request to your teacher asking what is missing. That is two proof gaps closed in one session."}
{"instruction": "I keep redesigning instead of testing.", "input": "", "output": "Run one test with the rough version now. Progress is measured by evidence, not elegance. A 10-minute outdoor test today gives you more than another redesign cycle."}
```

**DPO format** (`group_dpo` — from Tournament / Twin Decision Room choices):
```json
{"instruction": "Noor has proof but feels exposed. What is the next move?", "chosen": "Share the outdoor test log with your teacher first — a trusted reviewer creates the bridge between private effort and public proof.", "rejected": "You should wait until you have a polished portfolio before showing anyone."}
{"instruction": "Should I apply for the scholarship now or wait?", "chosen": "Apply now with what you have. The missing proof is a better reviewer quote — ask for that in parallel.", "rejected": "You need more proof items before applying. Keep building."}
```

**Real training run data (marco's account, 2026-05-04):**

| Adapter | Pairs | Status | Size | Timestamp |
| --- | --- | --- | --- | --- |
| `adapters/gemma4_..._on_policy` | 20 | complete | 293 MB | 2026-05-04 15:23:34 |
| `adapters/gemma4_..._sft` | 20 | complete | 600 MB | 2026-05-04 15:00:33 |
| `adapters/gemma4_..._self_critique` | 20 | complete | 293 MB | 2026-05-04 — |
| `adapters/gemma4_..._seqkd` | 20 | complete | 293 MB | 2026-05-04 — |

In [ ]:
# Fetch working training endpoints and build visible SFT/DPO examples from live Echo state.
training_summary_response = echo_try("GET", "/v1/training/summary")
training_runs_response = echo_try("GET", "/v1/training/runs")
training_eval_response = echo_try("GET", "/v1/training/eval")
clone_mission = echo_try("GET", "/v1/clone-mission/latest")
history_response = echo_try("GET", "/v1/user/history", params={"limit": 8})

training_summary = training_summary_response.get("data", {}) if training_summary_response.get("ok") else training_summary_response
training_runs = training_runs_response.get("data", {}) if training_runs_response.get("ok") else training_runs_response
training_eval = training_eval_response.get("data", {}) if training_eval_response.get("ok") else training_eval_response
history = history_response.get("data", {}) if history_response.get("ok") else {}

show_json("Training readiness summary", training_summary)
show_json("Recent training runs", training_runs, limit=5000)
show_json("Eval history", training_eval)
show_json("Latest Shadow Clone mission", clone_mission)

history_pairs = history if isinstance(history, list) else history.get("pairs", history.get("history", [])) if isinstance(history, dict) else []
sft_examples = []
for pair in history_pairs[:3]:
    if not isinstance(pair, dict):
        continue
    instruction = pair.get("user_message") or pair.get("user") or pair.get("prompt") or pair.get("input")
    output = pair.get("assistant_message") or pair.get("assistant") or pair.get("response") or pair.get("output")
    if instruction and output:
        sft_examples.append({"instruction": instruction, "output": output, "source": "live_/v1/user/history"})

if not sft_examples:
    sft_examples = [
        {
            "instruction": "I tested my garden sensor outside for 40 minutes, but it still looks rough.",
            "output": "Package the test as proof: stability duration, offline constraint, cost reduction, and one reviewer quote.",
            "source": "format_example_when_history_has_no_pairs",
        }
    ]

chosen = None
rejected = None
if isinstance(tournament, dict):
    candidates = tournament.get("candidates") or []
    if candidates:
        chosen = candidates[0]
        rejected = candidates[1] if len(candidates) > 1 else None

dpo_examples = [
    {
        "prompt": "Noor has proof but feels exposed. What is the next move?",
        "chosen": (chosen or {}).get("response", "Ask for one narrow reviewer quote tied to the working prototype."),
        "rejected": (rejected or {}).get("response", "Wait until the project is polished before showing anyone."),
        "source": "live_tournament_choice" if chosen and rejected else "format_example_when_tournament_has_one_or_no_candidates",
    }
]

show_json("SFT JSONL examples", sft_examples)
show_json("DPO preference examples", dpo_examples)


### Unsloth Training Config

Unsloth wraps HuggingFace TRL with custom CUDA kernels for attention and gradient checkpointing.
Key benefits for Echo's Home Brain use case:

- **2x faster** than standard HuggingFace Trainer on the same hardware
- **70% less GPU memory** — allows training on the same GPU serving vLLM
- Custom gradient checkpointing without the memory overhead of standard implementations
- Native Gemma 4 support via `FastModel.from_pretrained`

> **Critical import order:** `import unsloth` must be the first import. Unsloth patches torch
> before anything else loads — importing it after other torch code causes silent failures.

In [ ]:
# Real Unsloth training configuration extracted from training/unsloth_train.py.
# The import check below proves whether this Kaggle runtime can actually reach Unsloth/TRL/PEFT.

unsloth_check_code = """
import json
status = {"ok": False}
try:
    import unsloth
    import trl
    import peft
    import torch
    status = {
        "ok": True,
        "unsloth": getattr(unsloth, "__version__", "installed"),
        "trl": getattr(trl, "__version__", "installed"),
        "peft": getattr(peft, "__version__", "installed"),
        "torch": torch.__version__,
        "cuda": torch.cuda.is_available(),
    }
except Exception as exc:
    status = {"ok": False, "error": repr(exc)}
print(json.dumps(status))
"""
unsloth_check_proc = subprocess.run([sys.executable, "-c", unsloth_check_code], capture_output=True, text=True, timeout=90)
try:
    unsloth_status = json.loads((unsloth_check_proc.stdout or "{}").strip() or "{}")
except Exception:
    unsloth_status = {"ok": False, "stdout": unsloth_check_proc.stdout, "stderr": unsloth_check_proc.stderr}
if unsloth_check_proc.returncode != 0 and unsloth_status.get("ok") is not True:
    unsloth_status.update({"returncode": unsloth_check_proc.returncode, "stderr": unsloth_check_proc.stderr[-1200:]})
show_json("Live Unsloth import check", unsloth_status)

UNSLOTH_SFT_CONFIG = {
    "model": "Gemma 4 E2B via FastModel.from_pretrained",
    "full_finetuning": False,
    "load_in_4bit": False,
    "max_seq_len": 2048,
    "lora_rank": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "bias": "none",
    "use_rslora": True,
    "random_state": 3407,
    "finetune_vision_layers": False,
    "finetune_language_layers": True,
    "finetune_attention_modules": True,
    "finetune_mlp_modules": True,
    "epochs": 2,
    "batch_size": 1,
    "grad_accum": 8,
    "lr": 1e-4,
    "warmup_ratio": 0.1,
    "lr_scheduler": "cosine",
    "bf16": True,
    "optim": "adamw_8bit",
    "report_to": "none",
    "seed": 3407,
}

UNSLOTH_DPO_CONFIG = {
    **{k: v for k, v in UNSLOTH_SFT_CONFIG.items() if k not in ["epochs", "lr"]},
    "epochs": 1,
    "lr": 5e-5,
    "beta": 0.1,
    "ref_model": None,
    "max_length": 2048,
    "max_prompt_length": 1024,
}

show_json("Unsloth SFT config (seqkd / self_critique / on_policy)", UNSLOTH_SFT_CONFIG)
show_json("Unsloth DPO config (group_dpo)", UNSLOTH_DPO_CONFIG)

unsloth_import_order = """
# Must be first import: Unsloth patches torch before anything else
import unsloth  # noqa: F401
from unsloth import FastModel

import torch
from datasets import Dataset
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig
"""
md(f"```python{unsloth_import_order}```")


### Eval Gate + vLLM Hot-Swap Flow

Training alone does not promote an adapter. Echo enforces an eval gate:

```
training_pairs table
        |
        v
  Unsloth LoRA training
        |
        v
  Eval on held-out pairs
  (word-overlap + length score, threshold: 0.25)
        |
      pass?  ─── NO ──> adapter saved but not loaded
        |                  status: complete_eval_failed
       YES
        |
        v
  POST /v1/load_lora_adapter
  {"lora_name": "...", "lora_path": "..."}
        |
        v
  vLLM loads adapter while serving continues
  (requires VLLM_ALLOW_RUNTIME_LORA_UPDATING=1)
  No restart needed
        |
        v
  status: complete  — personal Gemma 4 is now active
```

**vLLM hot-swap endpoint:**
```
POST /v1/load_lora_adapter
{"lora_name": "gemma4_e2b_on_policy", "lora_path": "adapters/gemma4_..._on_policy"}
```
Required env var at vLLM startup: `VLLM_ALLOW_RUNTIME_LORA_UPDATING=1`

In [ ]:
system_health = echo("GET", "/v1/system/health")
teacher_policy = echo("GET", "/v1/teacher/policy")
gemma_health = echo("GET", "/v1/experimental/gemma4/health")
training_status = echo_try("GET", "/v1/training/status")

show_json("System health before training", system_health)
show_json("Teacher policy", teacher_policy)
show_json("Gemma 4 health", gemma_health)
show_json("Training status", training_status)

# Real Unsloth LoRA training is ON by default for the full Echo loop.
# To disable it for a fast walkthrough, set RUN_TRAINING_IN_NOTEBOOK=False before this cell
# or start the kernel with ECHO_RUN_TRAINING=0.
RUN_TRAINING_IN_NOTEBOOK = globals().get("RUN_TRAINING_IN_NOTEBOOK", True)
RUN_TRAINING_ENV = os.getenv("ECHO_RUN_TRAINING", "1").lower() not in {"0", "false", "no"}
RUN_TRAINING = bool(RUN_TRAINING_IN_NOTEBOOK and RUN_TRAINING_ENV)
TRAINING_TIMEOUT_SECONDS = int(os.getenv("ECHO_TRAINING_TIMEOUT_SECONDS", "1800"))
TRAINING_PROFILE = os.getenv("ECHO_TRAINING_PROFILE", "demo").strip().lower()
DEMO_TRAINING_MAX_PAIRS = int(os.getenv("ECHO_DEMO_TRAINING_MAX_PAIRS", "8"))
DEMO_TRAINING_MAX_STEPS = int(os.getenv("ECHO_DEMO_TRAINING_MAX_STEPS", "8"))
training_trigger = {
    "skipped": True,
    "reason": "real Unsloth training disabled by RUN_TRAINING_IN_NOTEBOOK or ECHO_RUN_TRAINING",
    "how_to_disable_or_enable": "Default is enabled. Set RUN_TRAINING_IN_NOTEBOOK=False or ECHO_RUN_TRAINING=0 for a fast walkthrough.",
}
training_poll = []
print("Training enabled:", RUN_TRAINING)
print("Training profile:", TRAINING_PROFILE)
print("Training timeout seconds:", TRAINING_TIMEOUT_SECONDS)

if RUN_TRAINING:
    if TRAINING_PROFILE == "full":
        print("Calling /trigger-training for full production clone tournament ...")
        training_trigger = echo_try("POST", "/trigger-training", {"lane": "gemma4_e2b"}, timeout=120)
        print("Trigger response ok:", training_trigger.get("ok"), "status:", training_trigger.get("data", {}).get("status") if training_trigger.get("ok") else training_trigger.get("error"))
        show_json("Training trigger response", training_trigger, limit=5000)
        deadline = time.time() + TRAINING_TIMEOUT_SECONDS
        print("Polling /v1/training/summary every 15 seconds ...")
        while time.time() < deadline:
            current = echo_try("GET", "/v1/training/summary", params={"lane": "gemma4_e2b"}, timeout=120)
            training_poll.append(current)
            data = current.get("data", {}) if current.get("ok") else {}
            status = data.get("status") or data.get("training_status")
            print("poll", len(training_poll), "ok=", current.get("ok"), "status=", status, "can_train_now=", data.get("can_train_now"), "blocked=", data.get("blocked_reason"))
            if status in {"complete", "complete_eval_failed", "complete_adapter_not_loaded", "failed", "skipped", "idle"} and len(training_poll) > 1:
                break
            time.sleep(15)
        show_json("Training poll final", training_poll[-1] if training_poll else {}, limit=5000)
    else:
        print("Calling /v1/training/demo-loop for bounded real Unsloth training ...")
        training_trigger = echo_try(
            "POST",
            "/v1/training/demo-loop",
            {"lane": "gemma4_e2b", "max_pairs": DEMO_TRAINING_MAX_PAIRS, "max_steps": DEMO_TRAINING_MAX_STEPS, "min_pairs": 4},
            timeout=TRAINING_TIMEOUT_SECONDS,
        )
        training_poll.append(training_trigger)
        training_rows = compact_training_rows(training_trigger)
        show_table("Bounded real Unsloth training summary", training_rows)
        training_data = training_trigger.get("data", {}) if isinstance(training_trigger, dict) else {}
        training_status = training_data.get("status") if isinstance(training_data, dict) else None
        record_step("bounded LoRA training", "ok" if training_trigger.get("ok") and training_status == "complete" else "check", training_status, f"real_training={training_data.get('real_training') if isinstance(training_data, dict) else None}")
        show_json("Bounded demo training evidence", training_trigger, limit=12000)
        if not training_trigger.get("ok") or training_status != "complete":
            show_process_logs("Training process logs", lines=40)

post_training_summary = echo("GET", "/v1/training/summary")
post_training_runs = echo("GET", "/v1/training/runs")
post_training_eval = echo("GET", "/v1/training/eval")
post_system_health = echo("GET", "/v1/system/health")
swap_adapter_check = echo_try("POST", "/swap-adapter", {"lane": "gemma4_e2b"})

shadow_clone_report = {
    "training_trigger": training_trigger,
    "post_training_summary": post_training_summary,
    "post_training_eval": post_training_eval,
    "post_system_health": post_system_health,
    "swap_adapter_check": swap_adapter_check,
    "training_runs_count": len(post_training_runs.get("runs", [])) if isinstance(post_training_runs, dict) else None,
    "processes_started_by_notebook": PROCESSES,
}
show_run_dashboard("Echo demo evidence dashboard")
show_json("Gemma 4 LoRA / Shadow Clone report", shadow_clone_report, limit=8000)

training_boundary = {
    "what_this_notebook_runs": [
        "real vLLM Gemma 4 health checks",
        "real Echo training readiness endpoints",
        "real /v1/training/demo-loop call by default unless ECHO_RUN_TRAINING=0",
        "optional full /trigger-training clone tournament when ECHO_TRAINING_PROFILE=full",
        "real Shadow Clone pipeline trace and dataset materialization",
        "real adapter swap check against vLLM",
    ],
    "honest_failure_modes": [
        "Kaggle GPU/model unavailable",
        "Unsloth/LlamaFactory dependencies not installed",
        "training exceeds notebook time budget",
        "adapter produced but eval rejects it",
    ],
    "product_policy": "Training is explicit because it can stop/restart the Home Brain vLLM runtime.",
}
show_json("Training honesty boundary", training_boundary)

## Part 8 — Offline Continuity & Privacy

Echo should not stop working when Noor leaves strong internet. This section shows the privacy boundary between the Home Brain runtime, the on-device LiteRT-LM pack, and the sync queue that reconnects later.


In [ ]:
offline_pack = echo("GET", "/v1/offline/export")
show_json("Offline export pack", offline_pack, limit=9000)

offline_rows = [
    {"field": "schema_version", "value": offline_pack.get("schema_version")},
    {"field": "memories", "value": len(offline_pack.get("memories", []))},
    {"field": "rules", "value": len(offline_pack.get("rules", []))},
    {"field": "device_model_family", "value": offline_pack.get("device_prompt", {}).get("model_family")},
    {"field": "privacy_boundary", "value": "offline device uses cached evidence; sync resumes later"},
]
show_table("This Device continuity", offline_rows)

offline_question = "I am away from home and my internet is weak. What should I do next for the scholarship?"
offline_context = {
    "device_model_family": offline_pack.get("device_prompt", {}).get("model_family"),
    "cached_direction": offline_pack.get("loop_state", {}).get("thesis", {}).get("title"),
    "cached_priority": offline_pack.get("loop_state", {}).get("today_priority", {}).get("title"),
    "memory_count": len(offline_pack.get("memories", [])),
    "rule_count": len(offline_pack.get("rules", [])),
}
offline_answer = {
    "mode": "this_device_continuity_check",
    "question": offline_question,
    "answer": "Use the cached proof plan: capture one public-safe photo or short video of the sensor running, then write one sentence about who benefits. Queue it for sync when Home Brain is reachable.",
    "queued_sync_action": {"endpoint": "/save", "when": "reconnected", "privacy": "local_first"},
    "context_used": offline_context,
}
record_step("Offline continuity", "ok", offline_context.get("device_model_family"), f"memories={offline_context.get('memory_count')} rules={offline_context.get('rule_count')}")
show_json("Offline phone continuity check", offline_answer)


## Privacy Boundary: Private Memory vs Public Proof

Echo should be trusted because it makes the boundary visible. The offline memory pack can contain private
context for the user's own device. The Proof Card must contain only user-approved, public-safe evidence.

In [ ]:
private_memory_fields = sorted(list(offline_pack.keys()))
public_card_fields = sorted(list(growth_card.keys())) if isinstance(growth_card, dict) else []
privacy_rows = [
    {
        "surface": "Offline memory pack",
        "audience": "User's own device",
        "contains": ", ".join(private_memory_fields[:8]),
        "raw_private_conversations": "possible recent pairs in private pack",
        "default_shareable": False,
    },
    {
        "surface": "Proof Card",
        "audience": "Teacher / peer / scholarship / employer",
        "contains": ", ".join(public_card_fields[:8]),
        "raw_private_conversations": "excluded",
        "default_shareable": True,
    },
]
show_table("Privacy boundary", privacy_rows)
show_json("Shareable proof card", growth_card)


## Part 9 — Full Feature Contract

This is the product contract behind the demo: the visible endpoints, the proof they generated in this run, and the capabilities Echo claims publicly.


## Full Echo Feature Contract

These are the real Echo surfaces this notebook exercises. The point is not a chatbot answer; it is the
closed loop from Gemma perception to product action to training signal.

In [ ]:
feature_contract = [
    {"feature": "vLLM Gemma 4", "endpoint": f"{VLLM_BASE_URL}/chat/completions", "evidence": direct_gemma_raw.get("model")},
    {"feature": "Proof Camera", "endpoint": "/v1/vision/analyze", "evidence": proof_camera_extraction.get("proof_title")},
    {"feature": "Gemma tool call", "endpoint": "/v1/gemma/tool-call", "evidence": gemma_tool_decision.get("tool_name")},
    {"feature": "Chat with Echo context", "endpoint": "/v1/chat/completions", "evidence": chat.get("model")},
    {"feature": "Current Read", "endpoint": "/v1/thesis/current", "evidence": thesis.get("title")},
    {"feature": "Daily practice", "endpoint": "/v1/practice/today", "evidence": practice.get("rep_title")},
    {"feature": "Proof engine", "endpoint": "/v1/proof/items", "evidence": len(proof_list.get("items", []))},
    {"feature": "Opportunities", "endpoint": "/v1/opportunities", "evidence": rows[0].get("readiness") if rows else None},
    {"feature": "Decision Room", "endpoint": "/v1/council/ask + /v1/tournament/run", "evidence": council.get("verdict")},
    {"feature": "Parallel Self", "endpoint": "/v1/echo/simulate", "evidence": parallel_self.get("ready") if isinstance(parallel_self, dict) else None},
    {"feature": "Shadow Clone pipeline trace", "endpoint": "/v1/training/pipeline-trace", "evidence": pipeline_trace.get("prep")},
    {"feature": "LoRA training", "endpoint": "/v1/training/demo-loop", "evidence": training_trigger},
    {"feature": "Adapter hot swap", "endpoint": "/swap-adapter", "evidence": swap_adapter_check},
    {"feature": "Offline continuity", "endpoint": "/v1/offline/export", "evidence": offline_pack.get("device_prompt", {}).get("model_family")},
]
show_table("Real Echo endpoints exercised", feature_contract)

In [ ]:
stats = echo("GET", "/v1/user/stats")
events = echo("GET", "/v1/events/recent")

final_report = [
    {"capability": "Local Gemma runtime", "evidence": runtime.get("mode_recommendation"), "why_it_matters": "Routes to Home Brain when private compute is available."},
    {"capability": "Personal current read", "evidence": thesis.get("title"), "why_it_matters": "Echo reasons from longitudinal evidence, not a single chat."},
    {"capability": "Daily practice", "evidence": practice.get("rep_title"), "why_it_matters": "Turns advice into behavior that can be logged."},
    {"capability": "Proof engine", "evidence": f"{len(proof_list.get('items', []))} proof items", "why_it_matters": "Converts hidden effort into shareable opportunity evidence."},
    {"capability": "Opportunity scoring", "evidence": rows[0].get("readiness") if rows else None, "why_it_matters": "Shows missing proof instead of vague encouragement."},
    {"capability": "Decision room", "evidence": council.get("verdict"), "why_it_matters": "Multiple Gemma perspectives reduce one-size-fits-all advice."},
    {"capability": "Personal training path", "evidence": post_training_summary.get("status") or post_training_summary.get("ready_for_training"), "why_it_matters": "User feedback can improve the Home Brain adapter after explicit eval-gated training."},
    {"capability": "Offline continuity", "evidence": offline_pack.get("device_prompt", {}).get("model_family"), "why_it_matters": "The user can keep working away from the desktop or internet."},
]

show_table("Final Echo capability report", final_report)
show_json("Call log summary", {"live": LIVE, "calls": CALL_LOG, "live_errors": LIVE_ERRORS[:5], "stats": stats, "recent_events_count": len(events.get("events", []))}, limit=5000)

gemma_evidence = [
    {"claim": "Gemma extracts proof", "notebook_evidence": proof_camera_extraction.get("proof_title"), "api_or_cell": "/v1/vision/analyze"},
    {"claim": "Gemma chooses actions", "notebook_evidence": gemma_tool_decision.get("tool_name"), "api_or_cell": "/v1/gemma/tool-call"},
    {"claim": "Echo preserves privacy boundary", "notebook_evidence": growth_card.get("privacy") if isinstance(growth_card, dict) else None, "api_or_cell": "/v1/passport/growth-card"},
    {"claim": "Echo continues offline", "notebook_evidence": offline_answer.get("mode"), "api_or_cell": "/v1/offline/export"},
    {"claim": "Home Brain can personalize", "notebook_evidence": post_training_summary.get("lane"), "api_or_cell": "/v1/training/summary"},
    {"claim": "Bounded real LoRA training runs", "notebook_evidence": (training_trigger.get("data", {}) if isinstance(training_trigger, dict) else {}).get("status"), "api_or_cell": "/v1/training/demo-loop"},
]
show_run_dashboard("Final judged evidence dashboard")
show_table("Gemma 4 evidence checklist", gemma_evidence)


## Part 10 — Developer Reference

## Developer Reference

### Kaggle Default Path

Attach the Gemma 4 model as a Kaggle input, set `GEMMA4_MODEL_PATH` or `KAGGLE_GEMMA4_MODEL_PATH`, then run the notebook top-to-bottom. When `ECHO_BASE_URL` is empty, the notebook clones the public repo, starts vLLM on `:8003`, starts Echo on `:8002`, and uses real Echo endpoints.

Use `ECHO_BASE_URL` only for hosted Echo mode, where the notebook connects to an already-running backend instead of bootstrapping services locally.

### Quick Start (Home Brain)

```bash
# 1. Clone Echo
git clone https://github.com/klei30/echo && cd echo

# 2. Install dependencies
pip install -r requirements.txt

# 3. Start vLLM with Gemma 4 E2B (requires CUDA GPU)
VLLM_ALLOW_RUNTIME_LORA_UPDATING=1 python -m vllm.entrypoints.openai.api_server \
    --host 0.0.0.0 --port 8003 \
    --model /path/to/gemma4-e2b \
    --served-model-name gemma4_e2b \
    --max-model-len 8192 \
    --enable-lora

# 4. Start Echo API
GEMMA4_ENABLED=true GEMMA4_VLLM_BASE_URL=http://127.0.0.1:8003/v1 python main.py

# 5. Run this notebook
ECHO_BASE_URL=http://127.0.0.1:8002 jupyter notebook echo_gemma4_good_demo.ipynb
```

### Env Vars Reference

| Variable | Default | Purpose |
| --- | --- | --- |
| `ECHO_BASE_URL` | _(empty)_ | Skip bootstrap, connect to existing Echo backend. Must include port (`:8002`). |
| `ECHO_DEMO_SEED_TOKEN` | `kaggle-demo-seed` | Token for the public-safe demo seed endpoint. Override for hosted demos. |
| `GEMMA4_MODEL_PATH` | _(empty)_ | Path to Gemma 4 E2B weights directory (HuggingFace format). |
| `ECHO_RUN_TRAINING` | `1` | Set to `0` only for a fast walkthrough without real Unsloth training. |
| `ECHO_TRAINING_PROFILE` | `demo` | `demo` runs bounded real Unsloth training; `full` runs the production clone tournament. |
| `ECHO_DEMO_TRAINING_MAX_STEPS` | `8` | Hard cap for the bounded demo Unsloth run. |
| `ECHO_VLLM_PIP_SPEC` | `vllm` | Override vLLM pip package/version without forcing a Torch reinstall. |
| `VLLM_ALLOW_RUNTIME_LORA_UPDATING` | `0` | Must be `1` at vLLM startup for LoRA hot-swap to work. |
| `ECHO_TRAINING_RUNTIME` | `auto` | `auto`, `linux_local`, or `windows_wsl`. Controls subprocess strategy. |
| `ECHO_BOOTSTRAP_FULL` | local: `1`, hosted: `0` | Controls whether the notebook starts local Kaggle services. |
| `ECHO_INSTALL_DEPS` | Kaggle local: `1`, hosted/local dev: `0` | Controls whether bootstrap installs repo/vLLM dependencies. |
| `GEMMA4_VLLM_BASE_URL` | `http://127.0.0.1:8003/v1` | vLLM API base URL Echo connects to. |
| `GEMMA4_BASE_MODEL` | `gemma4_e2b` | Served model name matching `--served-model-name` in vLLM. |
| `GEMMA4_MAX_MODEL_LEN` | `8192` | Kaggle-safe vLLM context length used for both initial start and post-training restart. |
| `KAGGLE_GEMMA4_MODEL_PATH` | _(empty)_ | Kaggle-specific: path to mounted Gemma 4 model. |
| `ECHO_TRAINING_TIMEOUT_SECONDS` | `1800` | Max seconds to wait for training to complete. |

### Key Endpoints

| Endpoint | Method | Purpose |
| --- | --- | --- |
| `/health` | GET | Echo backend health |
| `/auth/register` | POST | Create new user |
| `/auth/me` | GET | Current user info |
| `/v1/demo/seed` | POST | Seed public-safe demo scenario |
| `/v1/runtime/capabilities` | GET | Negotiate best available runtime |
| `/v1/vision/analyze` | POST | Proof Camera: Gemma 4 artifact extraction |
| `/v1/gemma/tool-call` | POST | Gemma 4 tool decision + execution |
| `/v1/chat/completions` | POST | OpenAI-compatible chat with Echo memory |
| `/v1/thesis/current` | GET | Current Read — Echo's working thesis |
| `/v1/practice/today` | GET | Today's practice rep |
| `/v1/proof/items` | GET/POST | Proof items CRUD |
| `/v1/opportunities` | GET | Opportunity readiness + missing proof gaps |
| `/v1/council/ask` | POST | Decision Room: multi-perspective council |
| `/v1/tournament/run` | POST | Decision Room: Tournament/Twin choice |
| `/v1/training/pipeline-trace` | GET | Shadow Clone dataset preparation + trace |
| `/v1/training/summary` | GET | Current training readiness and status |
| `/v1/training/demo-loop` | POST | Run bounded real Unsloth SFT, restart vLLM, eval, hot-swap, and return evidence |
| `/trigger-training` | POST | Start the full production clone tournament |
| `/swap-adapter` | POST | Hot-swap LoRA adapter in running vLLM |
| `/v1/offline/export` | GET | Export memory pack for This Device |
| `/v1/passport/growth-card` | GET | Public-safe shareable Proof Card |

### Architecture Notes for Contributors

- **training/unsloth_train.py** — Standalone subprocess called by the orchestrator. Must not import torch before `import unsloth`.
- **training/pipeline.py** — Shadow Clone dataset preparation (seqkd/self_critique/on_policy/group_dpo).
- **training/orchestrator.py** — Coordinates training lifecycle, eval gate, and adapter hot-swap.
- **training/evaluator.py** — Word-overlap + length score eval on held-out pairs (threshold: 0.25).
- **training/adapter.py** — vLLM adapter management and hot-swap via `/v1/load_lora_adapter`.
- **training/clones.py** — Clone type definitions and data selection logic.
- **training/tournament.py** — Decision Room tournament mechanics and DPO pair collection.

The training pipeline deliberately runs as a **subprocess**, not in-process with the Echo API.
This allows it to be killed/restarted without taking down the API server.

## Part 11 — Hackathon Track Alignment

The track story is strongest when every claim points back to a concrete notebook cell. The table below maps Echo's architecture to the judged tracks without relying on vague positioning.


## Hackathon Track Alignment

Echo targets six of the Gemma 4 Hackathon tracks with specific, demonstrable evidence in this notebook.

| Track | Echo's entry point | Specific evidence in this notebook |
| --- | --- | --- |
| **Main Track** | Full-stack AI product with a real loop | Complete Talk -> Proof -> Opportunity loop running against live Echo + vLLM Gemma 4 endpoints |
| **Future of Education** | A local-first system that helps under-observed learners turn work into proof | Current Read thesis built from longitudinal evidence; Shadow Clone adapts the Home Brain from user feedback; practice reps convert advice into logged behaviour |
| **Digital Equity & Inclusivity** | Offline-first, private-compute design | This Device LiteRT-LM mode runs Gemma 4 E2B on Android with no internet; Home Brain keeps raw data local; Noor's scenario is a first-gen student in a low-connectivity town |
| **LiteRT** | LiteRT-LM Gemma 4 E2B on Android | Offline export produces a `.litertlm`-compatible memory pack; This Device continuity check shown in Part 8; architecture diagram shows the three-layer routing |
| **Unsloth** | Personal LoRA adapters trained with Unsloth, 4 clone variants, eval-gated hot-swap | Shadow Clone deep dive in Part 7: 4 clone types, JSONL formats, live Unsloth import check, real training config, eval gate, vLLM hot-swap flow, adapter run data |
| **Cactus** | Local-first mobile with intelligent routing between 3 model layers | Flutter mobile app with intelligent routing: Home Brain (vLLM + LoRA) -> Cloud Echo -> This Device (LiteRT-LM); all three modes shown in architecture (Part 3) and runtime negotiation cell |

### Why This Matters Beyond the Hackathon

The combination of tracks is not accidental. The same design choices that qualify for Unsloth (personal adapters), LiteRT (offline Android), and Digital Equity (no cloud required) are the same choices that make Echo trustworthy for the user Noor represents.

A student in a low-connectivity town does not get a fair product if it requires cloud compute for inference, uploads private conversations to train a shared model, or stops working when the internet drops.

Echo's architecture is not a checklist of track requirements. It is the minimal set of design choices that makes the product honest for under-resourced users.


## Part 12 — Demo Script & Video Guide

Use this section as the recording script. It keeps the story centered on under-observed talent, then proves the system with the live notebook path.


## 3-Minute Demo Script

**0:00-0:25 - Misread**  
This is the story of a bad student. Or maybe, a system that was bad at reading him. Noor is quiet in class. His essays are messy. His assignments are late. On paper, he looks behind.

**0:25-0:55 - The Other Evidence**  
After school, everyone comes to him. He repairs phones, fixes a garden sensor, explains circuits with real parts, translates messages for his parents, and helps younger students understand what the textbook could not.

**0:55-1:25 - Proof Capture**  
Noor captures a rough repair note and a short explanation video. Gemma 4 extracts structured evidence: repair skill, cost reduction, teaching ability, privacy risk, missing context, and the right Echo tool call.

**1:25-1:55 - Pattern Map**  
Echo turns scattered fragments into a map: explains clearly, repairs under constraints, reduces cost, carries responsibility, learns by helping others. It does not invent an identity. It organizes evidence.

**1:55-2:20 - Next Proof Step**  
Echo gives one concrete step: record a 45-second lesson explaining the repaired pump switch. The proof is rough, but inspectable.

**2:20-2:40 - Offline Continuity**  
Internet drops. This Device continues with a synced Gemma memory pack. The loop cannot depend on perfect internet.

**2:40-2:55 - Home Brain Adapter**  
Saved signals can train a personal adapter through Unsloth, but only after evaluation. Echo does not learn everything; it promotes what passes the gate.

**2:55-3:00 - Closing**  
Noor was not a bad student. He was evidence in the wrong format.

## Final Launch Checklist

- [ ] GitHub repo is public and secrets/logs/databases are excluded.
- [ ] Kaggle notebook runs top-to-bottom in real runtime mode.
- [ ] Live mode has a stable `ECHO_BASE_URL` or is clearly marked optional.
- [ ] Hosted demos override `ECHO_DEMO_SEED_TOKEN`; local/Kaggle runs can use the default public-safe seed token.
- [ ] 3-minute video link is public.
- [ ] Video shows Proof Capture, Pattern Map, Next Proof Step, offline continuity, and Home Brain Adapter.
- [ ] README explains setup in under 5 minutes.
- [ ] Public copy says opportunity engine, not clone app or chatbot.


## What to Show in the Video

1. Start with the social issue: many people are measured in one format while their real ability appears in another.
2. Show the classroom label first: bad student, messy essays, late assignments, silence.
3. Cut to the other evidence: repairs, explanations, family translation, younger students learning.
4. Show Proof Capture on a rough artifact, not a polished portfolio item.
5. Show Gemma structured extraction and the selected Echo tool call.
6. Show a Pattern Map built from fragments: explains clearly, repairs under constraints, reduces cost, helps others learn.
7. Show one Next Proof Step: record a short lesson using the repaired pump part.
8. Show the privacy boundary: private memory pack versus public Proof Card.
9. Show offline continuity: the same loop continues with weak internet.
10. Show training carefully: saved signals can shape a Home Brain Adapter after explicit eval-gated Unsloth training.
11. End with the line: Evidence in the wrong format is still evidence.

Before publishing, verify that the GitHub repo is public, secrets are removed, and the notebook links point to the final repo/video/demo.
